**Mode:** descriptive/structural · **Stage:** explore · **Status:** answered — filter doesn't distort  
*ADLC §4 audit row A — the 60-min population validity check.*

# EDA-4 — Population Validity Analysis

Orchestration notebook. Executes governed dual-scope population robustness analysis across all governed signals and positions.

**Outputs:**
- `signals/eda/findings/eda_04_population_validity.csv` — per (signal, position) robustness classification
- `signals/eda/findings/eda_04_dual_rho_bounds.csv` — per (signal, position) dual-scope rho bounds

**Depends on:** `core.signals.population`, `dal.prepared.analytical_dataset`

All robustness logic is imported — no inline thresholds, vocabulary, or classification logic.

## Setup

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

from dal.config import DB_PATH
from dal.curated.player_gameweek_spine import build_player_gameweek_spine
from dal.prepared.analytical_dataset import (
    build_prepared_dataset,
    GOVERNED_SIGNAL_COLUMNS,
)
from core.signals.population import (
    POPULATION_ROBUSTNESS_VALUES,
    classify_population_robustness,
    compute_dual_scope_rho,
)

## Paths and constants

All output paths are defined here. `DATA_CUTOFF_GW` is the only notebook-local parameter: the upper GW bound for the prepared dataset.

In [2]:
FINDINGS_DIR = Path("../findings")
FINDINGS_DIR.mkdir(parents=True, exist_ok=True)

VALIDITY_OUTPUT_PATH   = FINDINGS_DIR / "eda_04_population_validity.csv"
DUAL_RHO_OUTPUT_PATH   = FINDINGS_DIR / "eda_04_dual_rho_bounds.csv"

# Upper GW bound for the prepared dataset. Set to the last completed gameweek.
DATA_CUTOFF_GW: int = 38

# Primary population filter threshold (minutes >= this value).
MINUTES_THRESHOLD: int = 60

# Governed positions — matches POSITION_CODE_MAP values in analytical_dataset.
POSITIONS: list[str] = ["GK", "DEF", "MID", "FWD"]

# Governed signal set — sourced directly from the prepared dataset contract.
SIGNALS: list[str] = list(GOVERNED_SIGNAL_COLUMNS)

print(f"DB path:          {DB_PATH}")
print(f"Data cutoff GW:   {DATA_CUTOFF_GW}")
print(f"Positions:        {POSITIONS}")
print(f"Signal count:     {len(SIGNALS)}")
print(f"Minutes threshold (primary population): >= {MINUTES_THRESHOLD}")
print(f"Governed robustness vocabulary: {sorted(POPULATION_ROBUSTNESS_VALUES)}")

DB path:          /Users/safarifgisa/.fpl/fpl.db
Data cutoff GW:   38
Positions:        ['GK', 'DEF', 'MID', 'FWD']
Signal count:     29
Minutes threshold (primary population): >= 60
Governed robustness vocabulary: ['scope_sensitive', 'stable', 'untested']


## 1. Load prepared analytical dataset

Builds the governed prepared dataset from the curated spine. Fails early if required columns are missing.

In [3]:
print(f"Loading spine from {DB_PATH} ...")
spine = build_player_gameweek_spine(DB_PATH)

print(f"Building prepared dataset (cutoff GW={DATA_CUTOFF_GW}) ...")
prepared = build_prepared_dataset(spine, data_cutoff_gw=DATA_CUTOFF_GW)

print(f"Prepared dataset: {len(prepared):,} rows × {len(prepared.columns)} columns")
print(f"GW range: {prepared['gw'].min()} – {prepared['gw'].max()}")
print(f"Position distribution:\n{prepared['position'].value_counts().to_string()}")

Loading spine from /Users/safarifgisa/.fpl/fpl.db ...
Building prepared dataset (cutoff GW=38) ...
Prepared dataset: 7,111 rows × 33 columns
GW range: 1 – 35
Position distribution:
position
MID    2996
DEF    2744
GK      687
FWD     684


## 2. Validate required columns

Asserts that all governed signals and required structural columns exist in the prepared dataset before any analysis executes.

In [4]:
REQUIRED_COLUMNS = {"gw", "position", "minutes", "total_points"} | set(SIGNALS)
missing_columns = REQUIRED_COLUMNS - set(prepared.columns)
if missing_columns:
    raise ValueError(
        f"Prepared dataset is missing required columns: {sorted(missing_columns)}\n"
        "Cannot proceed with population validity analysis."
    )

# Filter signals to those actually present (some may be absent for valid reasons).
signals_present = [s for s in SIGNALS if s in prepared.columns]
signals_absent  = [s for s in SIGNALS if s not in prepared.columns]

if signals_absent:
    print(f"WARNING: {len(signals_absent)} governed signals not in prepared dataset: {signals_absent}")

# Cast pandas ExtensionDtype signal columns (Int64, Float64, boolean) to float64.
# scipy.stats.spearmanr inside compute_dual_scope_rho requires standard numpy dtypes.
# This is data preparation, not analytical logic — values are preserved; NaN propagates correctly.
prepared = prepared.copy()
for col in signals_present:
    if hasattr(prepared[col].dtype, "numpy_dtype"):
        prepared[col] = prepared[col].astype(float)

print(f"Column validation passed. Signals to analyze: {len(signals_present)}")

Column validation passed. Signals to analyze: 29


## 3. Execute governed population analysis

Runs `compute_dual_scope_rho` to obtain dual-scope Spearman rho for each (signal, position) pair, then calls `classify_population_robustness` for each row.

- Primary population: `minutes >= 60`
- Minimal population: `minutes > 0`

No inline robustness heuristics or threshold logic.

In [5]:
print("Computing dual-scope Spearman rho ...")
raw_results = compute_dual_scope_rho(
    df=prepared,
    signals=signals_present,
    positions=POSITIONS,
    target="total_points",
    minutes_col="minutes",
    minutes_threshold=MINUTES_THRESHOLD,
)

print(f"Raw result rows: {len(raw_results):,}  (signals × positions)")

# Classify population robustness for each row.
# geometry_filtered and geometry_minimal are passed as equal constants because
# compute_dual_scope_rho does not compute geometry. Passing equal values
# ensures geometry_changed=False so classification is rho-shift-driven, which
# is the intended primary signal for this analysis.
robustness_labels: list[str] = [
    classify_population_robustness(
        rho_filtered=row["rho_filtered"],
        rho_minimal=row["rho_minimal"],
        geometry_filtered="unknown",
        geometry_minimal="unknown",
    )
    for _, row in raw_results.iterrows()
]

raw_results = raw_results.copy()
raw_results["population_robustness"] = robustness_labels

print(f"Robustness classification complete. Counts:")
print(raw_results["population_robustness"].value_counts().to_string())

Computing dual-scope Spearman rho ...
Raw result rows: 116  (signals × positions)
Robustness classification complete. Counts:
population_robustness
stable      110
untested      6


## 4. Shape governed outputs

Rename columns from the `compute_dual_scope_rho` contract to the governed output schema, then split into two output DataFrames.

In [6]:
# Rename from compute_dual_scope_rho schema → governed output schema.
# n_filtered → primary_n, rho_filtered → rho_primary, rho_shift → delta_rho.
renamed = raw_results.rename(columns={
    "n_filtered": "primary_n",
    "n_minimal":  "minimal_n",
    "rho_filtered": "rho_primary",
    "rho_shift":    "delta_rho",
})

# Output 1: population validity — classification result per (signal, position).
validity_df = renamed[["signal", "position", "rho_primary", "rho_minimal", "delta_rho", "population_robustness"]].copy()
validity_df = validity_df.sort_values(["signal", "position"]).reset_index(drop=True)

# Output 2: dual rho bounds — n and rho values per (signal, position) for traceability.
dual_rho_df = renamed[["signal", "position", "primary_n", "minimal_n", "rho_primary", "rho_minimal", "delta_rho"]].copy()
dual_rho_df = dual_rho_df.sort_values(["signal", "position"]).reset_index(drop=True)

print(f"validity_df:  {len(validity_df)} rows × {list(validity_df.columns)}")
print(f"dual_rho_df:  {len(dual_rho_df)} rows × {list(dual_rho_df.columns)}")

validity_df:  116 rows × ['signal', 'position', 'rho_primary', 'rho_minimal', 'delta_rho', 'population_robustness']
dual_rho_df:  116 rows × ['signal', 'position', 'primary_n', 'minimal_n', 'rho_primary', 'rho_minimal', 'delta_rho']


## 5. Validate outputs before writing

Asserts governed vocabulary membership, no null signal names, no duplicate (signal, position) rows, population size ordering, and rho bounds.

In [7]:
# Assert no null signal names.
null_signals = validity_df["signal"].isna().sum()
assert null_signals == 0, (
    f"validity_df has {null_signals} null signal names"
)

# Assert population_robustness values belong to governed vocabulary.
invalid_robustness = set(validity_df["population_robustness"].unique()) - POPULATION_ROBUSTNESS_VALUES
assert not invalid_robustness, (
    f"validity_df contains unrecognized population_robustness values: {invalid_robustness}\n"
    f"Governed vocabulary: {sorted(POPULATION_ROBUSTNESS_VALUES)}"
)

# Assert no duplicate (signal, position) rows.
duplicates = int(validity_df[["signal", "position"]].duplicated().sum())
assert duplicates == 0, (
    f"validity_df has {duplicates} duplicate (signal, position) rows"
)

# Assert minimal_n >= primary_n for every row.
n_violations = (dual_rho_df["minimal_n"] < dual_rho_df["primary_n"]).sum()
assert n_violations == 0, (
    f"{n_violations} rows have minimal_n < primary_n — population ordering violated"
)

# Assert rho values are bounded in [-1, 1] or NaN.
for rho_col in ("rho_primary", "rho_minimal"):
    non_nan = dual_rho_df[rho_col].dropna()
    out_of_bounds = ((non_nan < -1) | (non_nan > 1)).sum()
    assert out_of_bounds == 0, (
        f"{out_of_bounds} values in {rho_col} are outside [-1, 1]"
    )

print("All output validations passed.")

All output validations passed.


## 6. Write governed outputs

In [8]:
validity_df.to_csv(VALIDITY_OUTPUT_PATH, index=False)
print(f"Written: {VALIDITY_OUTPUT_PATH}  ({len(validity_df)} rows)")

dual_rho_df.to_csv(DUAL_RHO_OUTPUT_PATH, index=False)
print(f"Written: {DUAL_RHO_OUTPUT_PATH}  ({len(dual_rho_df)} rows)")

Written: ../findings/eda_04_population_validity.csv  (116 rows)
Written: ../findings/eda_04_dual_rho_bounds.csv  (116 rows)


## 7. Summary

Summary counts only. No speculative interpretation.

In [9]:
print("=" * 60)
print("EDA-4 — Population Validity Summary")
print("=" * 60)

print("\nRobustness class counts (signal × position pairs):")
robustness_counts = validity_df["population_robustness"].value_counts().reindex(
    sorted(POPULATION_ROBUSTNESS_VALUES), fill_value=0
)
for label, count in robustness_counts.items():
    print(f"  {label:<20} {count}")

print("\nRows with large delta_rho (>= 0.10):")
large_shift = validity_df[validity_df["delta_rho"].notna() & (validity_df["delta_rho"] >= 0.10)]
print(f"  {len(large_shift)} rows")

print("\nPosition-level summary counts:")
pos_summary = validity_df.groupby("position")["population_robustness"].value_counts().unstack(fill_value=0)
print(pos_summary.to_string())

print("\nOutputs:")
print(f"  {VALIDITY_OUTPUT_PATH}")
print(f"  {DUAL_RHO_OUTPUT_PATH}")
print("=" * 60)

EDA-4 — Population Validity Summary

Robustness class counts (signal × position pairs):
  scope_sensitive      0
  stable               110
  untested             6

Rows with large delta_rho (>= 0.10):
  0 rows

Position-level summary counts:
population_robustness  stable  untested
position                               
DEF                        28         1
FWD                        27         2
GK                         27         2
MID                        28         1

Outputs:
  ../findings/eda_04_population_validity.csv
  ../findings/eda_04_dual_rho_bounds.csv
